# Lab 01 guiado · Importar y unir

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 1, ruta guiada*

Este cuaderno se abre directo desde GitHub. Lo que escribas se pierde al cerrar la pestaña. Si quieres conservarlo, descárgalo con Archivo, Descargar, Descargar .ipynb.

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#31708f"></td><td bgcolor="#d9edf7"><font color="#0b3d62" size="3"><b>Qué vamos a responder</b><br>• ¿Cómo se abre y se mira una tabla?<br>• ¿De dónde pueden venir los datos?<br>• ¿Qué se pierde o se duplica al unir dos tablas?</font></td></tr></table>

---
<br>

## Paso 1 de 10 · ¿Qué centrales tenemos?

**Qué vamos a responder.** Qué centrales tiene el sistema del curso y qué datos trae cada una.

**Qué hacer.** Corre la celda. La tabla se lee desde la carpeta de datos del repositorio del curso.

In [ ]:
# Paso 1
# La carpeta de datos del curso, en el repositorio.
DATOS = "https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/datos/"

import pandas as pd

# read_csv lee un archivo separado por comas y lo convierte en una tabla.
centrales = pd.read_csv(DATOS + "centrales.csv")
centrales

---
<br>

## Paso 2 de 10 · ¿Qué tamaño tiene la tabla?

**Qué vamos a responder.** Cuántas filas y columnas tiene la tabla y cuánta potencia suma.

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 2
# Tres preguntas que uno le hace a cualquier tabla apenas la abre.
print("Filas y columnas", centrales.shape)
print("Columnas        ", list(centrales.columns))
print("Potencia total  ", centrales["potencia_mw"].sum(), "MW")

---
<br>

## Paso 3 de 10 · Tu turno, filtra por tecnología

**Qué vamos a responder.** Cuántas centrales hay de una tecnología y cuáles son.

**Qué hacer.** Elige una tecnología en la lista y vuelve a correr la celda.

In [ ]:
# Paso 3
# La tabla deja solo las centrales de la tecnología elegida.
tecnologia = "hidro" #@param ["hidro", "solar", "eolica", "gas", "carbon", "diesel"]

filtradas = centrales[centrales["tecnologia"] == tecnologia]
print(f"✓ Hay {len(filtradas)} centrales de {tecnologia}")
filtradas

---
<br>

## Paso 4 de 10 · Tu turno, resume la potencia

**Qué vamos a responder.** Cuánta potencia instalada hay por tecnología o por región.

**Qué hacer.** Elige cómo agrupar en la lista y vuelve a correr la celda.

In [ ]:
# Paso 4
# Suma la potencia de las centrales que comparten el valor elegido.
agrupar_por = "tecnologia" #@param ["tecnologia", "region"]

import matplotlib.pyplot as plt
resumen = centrales.groupby(agrupar_por)["potencia_mw"].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(resumen.index, resumen.values, color="#4C78A8")
ax.set_title(f"Potencia instalada por {agrupar_por}")
ax.set_xlabel("Potencia instalada (MW)")
plt.show()

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>Ojo con esta cifra</b><br>Este parque de veinte centrales es inventado para el curso y no reproduce<br>la matriz chilena. En él la solar es segunda en potencia instalada y quinta<br>en energía generada, porque de noche no genera. Esa diferencia entre potencia<br>y energía es justo lo que este bloque quiere mostrar.<br><br>Los datos del curso son sintéticos. Estas cifras no sirven para hablar<br>del sistema eléctrico real.</font></td></tr></table>

---
<br>

## Paso 5 de 10 · ¿Cómo llega la generación de cada hora?

*En los pasos 5 a 8 solo se corre y se mira. Venga de donde venga el dato, termina en una tabla igual.*

**Qué vamos a responder.** Cómo viene la generación de cada central, hora por hora, durante 2024.

**Qué hacer.** Corre la celda. El archivo es grande y puede demorar unos segundos.

In [ ]:
# Paso 5
# parse_dates avisa que la columna fecha trae fechas y no texto.
generacion = pd.read_csv(DATOS + "generacion.csv", parse_dates=["fecha"])

print("Filas y columnas", generacion.shape)
print("Primer día      ", generacion["fecha"].min().date())
print("Último día      ", generacion["fecha"].max().date())
generacion.head()

---
<br>

## Paso 6 de 10 · ¿Y si los datos vienen en Excel?

**Qué vamos a responder.** Cómo se lee un Excel que trae más de una hoja.

**Qué hacer.** Corre la celda y mira la hoja de notas.

In [ ]:
# Paso 6
# Con sheet_name=None se traen todas las hojas del Excel de una vez.
hojas = pd.read_excel(DATOS + "centrales.xlsx", sheet_name=None)

print("Hojas del archivo", list(hojas.keys()))
hojas["notas"]

---
<br>

## Paso 7 de 10 · ¿Y si vienen de un servicio web?

**Qué vamos a responder.** Cómo se convierte en tabla la respuesta de un servicio web.

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 7
# Así se lee un servicio web de verdad. Aquí la dirección apunta a un archivo del curso.
import requests
respuesta = requests.get(DATOS + "precios_nudo.json").json()

print("Llaves de la respuesta", list(respuesta.keys()))

# json_normalize convierte la lista de registros en una tabla.
pd.json_normalize(respuesta["datos"]).head()

---
<br>

## Paso 8 de 10 · ¿Y si vienen de una base de datos?

**Qué vamos a responder.** Cómo se le pide a una base de datos un resumen ya calculado.

![La base de datos calcula y entrega solo el resumen](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo01/img/lab01_paso8_sql.png)

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 8
# Se baja el archivo de la base a Colab y se abre una conexión con él.
import sqlite3, urllib.request
urllib.request.urlretrieve(DATOS + "demanda.db", "demanda.db")
conexion = sqlite3.connect("demanda.db")

# El filtro se escribe en SQL y lo resuelve la base, así viaja mucho menos dato.
consulta = """
    SELECT region, SUM(mwh) AS demanda_total
    FROM demanda
    GROUP BY region
    ORDER BY demanda_total DESC
"""
pd.read_sql(consulta, conexion)

---
<br>

## Paso 9 de 10 · Tu turno, une generación con fichas

**Qué vamos a responder.** Cuánta generación se pierde al unir con una tabla de fichas incompleta.

![Unir con inner pierde las filas sin pareja](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo01/img/lab01_paso9_join.png)

**Qué hacer.** Elige cómo unir en la lista y vuelve a correr la celda. Compara la generación real con la unida.

In [ ]:
# Paso 9
# inner deja solo lo que está en las dos tablas.
# left conserva toda la generación.
como_unir = "inner" #@param ["inner", "left"]

# Un día de generación y una tabla de fichas a la que le faltan
# tres centrales.
un_dia = generacion[generacion["fecha"] == "2024-06-15"]
sin_ficha = ["Eólica Loma Fría", "Central Vega Azul", "Diésel Respaldo Cordillera"]
fichas = centrales[~centrales["central"].isin(sin_ficha)]

unido = pd.merge(un_dia, fichas, on="central", how=como_unir)
print("Generación real del día", round(un_dia["mwh"].sum(), 1), "MWh")
print("Generación unida       ", round(unido["mwh"].sum(), 1), "MWh")
print()
# Las centrales sin ficha quedan agrupadas como "sin ficha".
unido_con_ficha = unido.fillna({"tecnologia": "sin ficha"})
por_tecnologia = unido_con_ficha.groupby("tecnologia")["mwh"].sum().round(1)
print(por_tecnologia.sort_values(ascending=False))

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3">Inner deja fuera, sin avisar, las centrales que no tienen ficha.<br>Left conserva toda la generación y deja vacía la ficha que falta.<br>Por eso, antes y después de unir se cuentan las filas y se compara el total.</font></td></tr></table>

---
<br>

## Paso 10 de 10 · Tu turno, ¿qué pasa con fichas repetidas?

**Qué vamos a responder.** Qué le pasa al total cuando la tabla de fichas trae centrales repetidas.

![Una ficha repetida duplica las horas de su central](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo01/img/lab01_paso10_repetidas.png)

**Qué hacer.** Corre la celda con la casilla sin marcar y mira el total. Después márcala y vuelve a correrla.

In [ ]:
# Paso 10
# Marca la casilla para dejar una sola ficha por central.
quitar_repetidas = False #@param {type:"boolean"}

# Una tabla de fichas como llega en la práctica, con dos
# centrales repetidas.
fichas_repetidas = pd.concat([fichas, fichas.head(2)], ignore_index=True)
if quitar_repetidas:
    fichas_repetidas = fichas_repetidas.drop_duplicates(subset="central")

unido = pd.merge(un_dia, fichas_repetidas, on="central", how="left")
real = round(un_dia["mwh"].sum(), 1)
unida = round(unido["mwh"].sum(), 1)
print("Filas antes de unir", un_dia.shape[0], "y después", unido.shape[0])
print("Generación real", real, "MWh y generación unida", unida, "MWh")
if unida == real:
    print("✓ La generación unida calza con la real")

---
<br>

## Lo que lograste hoy

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#28a745"></td><td bgcolor="#d8f0df"><font color="#14532d" size="3">• Una tabla se mira primero por su tamaño, sus columnas y un total.<br>• Un filtro deja las filas que interesan y un resumen agrupa.<br>• Los datos pueden venir de CSV, Excel, un servicio web o una base de datos,<br>&nbsp;&nbsp;y siempre terminan en una tabla.<br>• Antes y después de unir se cuentan las filas y se compara el total.</font></td></tr></table>

Las otras formas de unir y el trabajo con archivos muy grandes están en [la ruta completa de este módulo](https://colab.research.google.com/github/RodrigoMSB/curso-python-minergia/blob/main/modulos/modulo01/lab.ipynb).